# Aula 2.2 — Feature Engineering com Databricks Feature Store 🚀

Nesta aula, vamos abordar um dos maiores desafios ao colocar modelos de "Machine Learning" em produção: a consistência entre o momento do treino e o momento da inferência, além de mitigar o "data leakage" (vazamento de dados).

## 1. O Problema de Consistência Treino-Inferência
Em sistemas tradicionais, o cientista de dados cria variáveis complexas durante o treinamento. No entanto, o time de engenharia precisa reescrever toda essa lógica para rodar em tempo real na API de produção. Esse descasamento gera erros graves e perda de performance do modelo ("Feature Drift").

A **Feature Store** do Databricks, governada pelo **Unity Catalog**, resolve isso atuando como um repositório centralizado de recursos:
1. **Fonte Única da Verdade:** O mesmo cálculo usado no treino é disponibilizado em baixa latência para a produção.
2. **"Point-in-time correctness":** Evita o "data leakage" usando o "time travel" do Delta Lake para garantir que o modelo só olhe para o passado exato do evento durante o treino.
3. **Qualidade de Dados:** Tabelas de recursos centralizadas garantem padronização de tipos e linhagem de dados nativa.

---

## Dataset Iris
Para focar puramente na sintaxe e arquitetura do `FeatureEngineeringClient`, utilizaremos o clássico "dataset" **Iris**. Como as "Feature Tables" do Unity Catalog exigem uma **Chave Primária**, adicionaremos programaticamente uma coluna de identificação única (`id_flor`).

In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
from sklearn.datasets import load_iris
from pyspark.sql.functions import col, expr, rand

# Catalogo e Esquema do dataset Iris
catalogo_origem = "workspace"
esquema_origem = "default"
tabela_iris_raw = f"{catalogo_origem}.{esquema_origem}.iris_dataset"

# Lendo os dados brutos que já estão no ambiente
df_iris = spark.table(tabela_iris_raw)

# Garanta que a tabela possua a chave primária (id_flor) e as colunas ajustadas.

# Registra o DataFrame atual como uma View temporária no Spark
df_iris.createOrReplaceTempView("vw_iris_raw")

# Executa a query SQL para construir a chave primária
df_iris_sql = spark.sql("""
    SELECT 
        CONCAT('flor_', ROW_NUMBER() OVER (ORDER BY (SELECT NULL))) AS id_flor,
        sepal_length,
        sepal_width,
        petal_length,
        petal_width,
        species
    FROM vw_iris_raw
""")

# Sobrescreve o DataFrame original com o resultado do SQL
df_iris_new = df_iris_sql

display(df_iris_new.limit(5))


id_flor,sepal_length,sepal_width,petal_length,petal_width,species
flor_1,5.1,3.5,1.4,0.2,Iris-setosa
flor_2,4.9,3.0,1.4,0.2,Iris-setosa
flor_3,4.7,3.2,1.3,0.2,Iris-setosa
flor_4,4.6,3.1,1.5,0.2,Iris-setosa
flor_5,5.0,3.6,1.4,0.2,Iris-setosa


In [0]:
from databricks import feature_engineering

# Definindo o destino das nossas features
nome_tabela_features = f"{catalogo_origem}.{esquema_origem}.iris_features_table"

# Instanciando o cliente de engenharia Databricks
fe = feature_engineering.FeatureEngineeringClient()

# Isolando apenas os recursos (Features) e a chave primária que vão para a store
# Removemos a coluna target (species) para que ela não seja exposta na tabela de recursos
df_features_only = df_iris_new.select("id_flor", "sepal_length", "sepal_width", "petal_length", "petal_width")

print(f" Gravando e registrando a Feature Table em: {nome_tabela_features}...")

# Criando e populando a tabela de recursos de forma centralizada
fe.create_table(
    name=nome_tabela_features,
    primary_keys=["id_flor"],
    df=df_features_only,
    schema=df_features_only.schema,
    description="Tabela de recursos morfológicos das flores Iris integrada à Feature Store do Unity Catalog."
)
print("Features salvas e sincronizadas com sucesso! Você já consegue visualizá-las pelo Catalog Explorer.")

 Gravando e registrando a Feature Table em: workspace.default.iris_features_table...


2026/06/23 20:31:47 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['id_flor'] of table 'workspace.default.iris_features_table' to NOT NULL.
2026/06/23 20:31:49 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['id_flor'] on table 'workspace.default.iris_features_table'.
2026/06/23 20:31:56 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'workspace.default.iris_features_table'.


Features salvas e sincronizadas com sucesso! Você já consegue visualizá-las pelo Catalog Explorer.


In [0]:
from pyspark.sql.functions import col

# 1. Carrega a tabela de recursos atual a partir da Feature Store para derivar a nova informação
df_existente = spark.table(nome_tabela_features)

print("Calculando a nova feature: razão morfológica da pétala (petal_ratio)...")

# Criando a nova variável baseada em colunas existentes (Razão = Comprimento / Largura)
# Mantemos obrigatoriamente a chave primária 'id_flor' no DataFrame resultante
df_nova_feature = df_existente.select(
    col("id_flor"),
    (col("petal_length") / col("petal_width")).alias("petal_ratio")
)

print(f"Atualizando a Feature Table no Unity Catalog: {nome_tabela_features}...")

# Gravando a nova característica usando o write_table
# O Databricks faz o merge inteligente baseado na chave primária ['id_flor']
fe.write_table(
    name=nome_tabela_features,
    df=df_nova_feature,
    mode="merge" # Realiza o update das linhas combinando as chaves primárias
)

print("Nova feature 'petal_ratio' adicionada e integrada com sucesso!")

# Visualizando o novo esquema atualizado
display(spark.table(nome_tabela_features).limit(5))

Calculando a nova feature: razão morfológica da pétala (petal_ratio)...
Atualizando a Feature Table no Unity Catalog: workspace.default.iris_features_table...
Nova feature 'petal_ratio' adicionada e integrada com sucesso!


id_flor,sepal_length,sepal_width,petal_length,petal_width,petal_ratio
flor_1,5.1,3.5,1.4,0.2,6.999999999999999
flor_2,4.9,3.0,1.4,0.2,6.999999999999999
flor_3,4.7,3.2,1.3,0.2,6.5
flor_4,4.6,3.1,1.5,0.2,7.5
flor_5,5.0,3.6,1.4,0.2,6.999999999999999


In [0]:
%sql
ALTER TABLE workspace.default.iris_features_table SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5');
ALTER TABLE workspace.default.iris_features_table DROP COLUMNS (petal_ratio);

In [0]:
%sql
SELECT * FROM workspace.default.iris_features_table

id_flor,sepal_length,sepal_width,petal_length,petal_width
flor_1,5.1,3.5,1.4,0.2
flor_2,4.9,3.0,1.4,0.2
flor_3,4.7,3.2,1.3,0.2
flor_4,4.6,3.1,1.5,0.2
flor_5,5.0,3.6,1.4,0.2
flor_6,5.4,3.9,1.7,0.4
flor_7,4.6,3.4,1.4,0.3
flor_8,5.0,3.4,1.5,0.2
flor_9,4.4,2.9,1.4,0.2
flor_10,4.9,3.1,1.5,0.1


In [0]:
%sql
DESCRIBE HISTORY workspace.default.iris_features_table

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-06-23T20:34:15.000Z,72226572122353,joaom.pinheiro@outlook.com,DROP COLUMNS,"Map(columns -> [""petal_ratio""])",null,List(4131393660666134),5057a1bb-2d9c-48ce-8381-159fa5f8a4f9,0623-192115-w3kksqg0-v2n,5,WriteSerializable,true,Map(),null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
5,2026-06-23T20:34:14.000Z,72226572122353,joaom.pinheiro@outlook.com,SET TBLPROPERTIES,"Map(properties -> {""delta.columnMapping.mode"":""name"",""delta.minReaderVersion"":""2"",""delta.minWriterVersion"":""5""})",null,List(4131393660666134),9625a0cb-1592-445f-b597-4ddd9fdbdfb7,0623-192115-w3kksqg0-v2n,4,WriteSerializable,true,Map(),null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
4,2026-06-23T20:33:39.000Z,72226572122353,joaom.pinheiro@outlook.com,MERGE,"Map(predicate -> [""(id_flor#14726 = id_flor#14433)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(4131393660666134),f6235b3e-43d5-45c3-8906-b0d9c9cf7cc3,0623-192115-w3kksqg0-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3899, numTargetBytesRemoved -> 2974, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 150, executionTimeMs -> 4252, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1345, numTargetRowsUpdated -> 150, numOutputRows -> 150, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 150, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2865)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
3,2026-06-23T20:33:32.000Z,72226572122353,joaom.pinheiro@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)",null,List(4131393660666134),146b0590-ceea-48be-a3c2-96e2466bcd7f,0623-192115-w3kksqg0-v2n,2,WriteSerializable,true,"Map(numFiles -> 0, numOutputRows -> 0, numOutputBytes -> 0)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
2,2026-06-23T20:31:56.000Z,72226572122353,joaom.pinheiro@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> Tabela de recursos morfológicos das flores Iris integrada à Feature Store do Unity Catalog., isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true, canMergeSchema -> true)",null,List(4131393660666134),4fad0aa5-7844-4071-a369-5e773d718c2a,0623-192115-w3kksqg0-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 150, numOutputBytes -> 2974)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
1,2026-06-23T20:31:49.000Z,72226572122353,joaom.pinheiro@outlook.com,CHANGE COLUMN,"Map(column -> {""name"":""id_flor"",""type"":""string"",""nullable"":false,""metadata"":{}})",null,List(4131393660666134),95d8b4cf-9f35-4aab-a724-501445f75893,0623-192115-w3kksqg0-v2n,0,WriteSerializable,false,Map(),null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-06-23T20:31:47.000Z,72226572122353,joaom.pinheiro@outlook.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> Tabela de recursos morfológicos das flores Iris integrada à Feature Store do Unity Catalog., isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.fo

### Qual a diferença entre uma Tabela Delta Comum e uma "Feature Table"?

Para quem olha de fora (via comandos `SELECT`), ambas parecem exatamente iguais. No entanto, ao registrar seus dados usando o `FeatureEngineeringClient` em vez de um simples `.write.saveAsTable()`

A tabela abaixo resume as diferenças fundamentais:

| Funcionalidade / Recurso | Tabela Delta Comum  | Feature Store |
| :--- | :--- | :--- |
| **Formato Físico** | Delta Lake (Arquivos Parquet) | Delta Lake (Arquivos Parquet) |
| **Chave Primária Obrigatória** | Não exigida. | **Obrigatória**. Define o índice de busca (ex: `id_flor`, `id_usuario`). |
| **Rastreabilidade de IA (Linhagem)** | Sabe apenas quais tabelas geraram os dados. | Sabe quais modelos, "notebooks" e "pipelines" consomem cada coluna. |
| **Prevenção de Vazamento de Dados** | Manual. Você precisa codificar os `JOINs` temporais na unha. | **Automatizada**. O "Point-in-time correctness" usa o "timestamp" para evitar "data leakage". |
| **Integração com APIs de Produção** | Exige engenharia para reescrever consultas SQL em tempo real. | **Nativa**. O "Model Serving" busca os dados sozinho em "Online Tables" de baixa latência. |

---

### O Cenário Prático na Empresa:

1. **Se você salva como Tabela Comum:** O cientista de dados cria a feature `media_compras_30d`. Se outro cientista precisar usar essa mesma variável em outro modelo, ele terá que adivinhar qual é o script de engenharia de dados, recalculá-la por conta própria (correndo o risco de usar regras diferentes) e gerar duplicidade de dados no "lake".
2. **Se você salva na Feature Store:** O recurso fica indexado no painel do **Catalog Explorer**. Qualquer pessoa da empresa pode pesquisar por `media_compras_30d`, entender quem a criou, ver o gráfico de linhagem e consumi-la instantaneamente com uma linha de código, garantindo governança absoluta e reuso de código.